In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
from scipy.stats import randint

In [ ]:
df = pd.read_csv("2025_DS2_HW1_data_train.csv")
print(df.info())

We see that we have 15 variables with "float64" data types and 4 with "object" data types.

In [ ]:
missing_counts = df.isnull().sum()
info_summary = pd.DataFrame({
    'Percentage Missing': (df.isnull().sum() / len(df)) * 100
})

print(missing_counts)
print(info_summary)

We printed out list of numbers of missing values as well as the percentage of missing values in the category.


In [ ]:
summary = df.describe()
print(summary)

We print our descriptive statistics.

In [ ]:
df_clean = df.dropna(subset=['booking_status'])
df_clean = df_clean.drop(columns=['Booking_ID'])
corr = df_clean.select_dtypes(include=['float64', 'int64']).corr()
booking_status_corr = corr['booking_status'].sort_values(ascending=False)

print(booking_status_corr)

We can see correlations with booking status. With the strongest correlations being lead time

In [ ]:

plt.figure(figsize=(7, 5))
# Fix: Assigned 'booking_status' to both x and hue, then set legend=False
sns.countplot(
    data=df_clean,
    x='booking_status',
    hue='booking_status',
    legend=False,
    palette='viridis'
)
plt.title('Distribution of Booking Status (0: Not Canceled, 1: Canceled)')
plt.show()
# B. Distributions of Key Features
# Lead Time and Average Price are usually strong predictors
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(df_clean['lead_time'].dropna(), kde=True, ax=axes[0], color='blue')
axes[0].set_title('Distribution of Lead Time')

sns.histplot(df_clean['avg_price_per_room'].dropna(), kde=True, ax=axes[1], color='green')
axes[1].set_title('Distribution of Average Price per Room')

plt.show()

plt.figure(figsize=(7, 5))
sns.barplot(
    x='no_of_children',
    y='booking_status',
    data=df_clean,
    errorbar=None,
    hue="no_of_children",
    legend=False,
    palette='mako'
)

plt.title('Distribution of Number of children')
plt.show()
# C. Relationships with the Target Variable

# Special Requests vs. Status
# Fix: Replaced 'ci' with 'errorbar' and assigned 'x' to 'hue'
plt.figure(figsize=(10, 6))
sns.barplot(
    x='no_of_special_requests',
    y='booking_status',
    data=df_clean,
    errorbar=None,
    hue='no_of_special_requests',
    legend=False,
    palette='mako'
)
plt.ylabel('Cancellation Rate')
plt.title('Cancellation Rate by Number of Special Requests')
plt.show()

# Market Segment vs. Status
# Fix: Assigned 'x' to 'hue' to satisfy the new requirement
plt.figure(figsize=(12, 6))
sns.countplot(
    x='market_segment_type',
    hue='market_segment_type',
    data=df_clean,
    palette='pastel',
    legend=False
)
# Note: If you want to see the breakdown of cancellation WITHIN the segment:
# sns.countplot(x='market_segment_type', hue='booking_status', data=df_clean, palette='pastel')

plt.title('Market Segment Count')
plt.xticks(rotation=45)
plt.show()

In plot 1 we see number of cancellations.

In [ ]:
num_features = ['lead_time', 'avg_price_per_room', 'no_of_special_requests', 'no_of_week_nights']
plt.figure(figsize=(16, 10))
for i, col in enumerate(num_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(data=df_clean, x='booking_status', y=col, palette='Set2')
    plt.title(f'{col} by Booking Status')
plt.tight_layout()
plt.show()
cat_features = ['market_segment_type', 'type_of_meal_plan', 'room_type_reserved']
plt.figure(figsize=(16, 18))
for i, col in enumerate(cat_features, 1):
    plt.subplot(3, 1, i)
    counts = df_clean.groupby([col, 'booking_status']).size().unstack(fill_value=0)
    proportions = counts.div(counts.sum(axis=1), axis=0)
    proportions.plot(kind='bar', stacked=True, ax=plt.gca(), color=['#66b3ff', '#ff9999'])
    plt.title(f'Proportion of Booking Status by {col}')
    plt.ylabel('Proportion')
    plt.legend(['Not Canceled', 'Canceled'], loc='upper right')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

TODO

In [ ]:
le = LabelEncoder()
for col in df_clean.select_dtypes(include=['object']).columns:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

# Impute missing values with median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df_clean), columns=df_clean.columns)

X = df_imputed.drop(columns=['booking_status'])
y = df_imputed['booking_status']

# Pearson Correlation
correlations = df_imputed.corr()['booking_status'].abs().drop('booking_status')

#Mutual Information
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X.columns)

#F-test Importance (Univariate importance)
# This calculates the ANOVA F-value for each feature
f_values, p_values = f_classif(X, y)
f_series = pd.Series(f_values, index=X.columns)

# --- METHOD 4: Random Forest Importance (Multivariate/Embedded) ---
#rf = RandomForestClassifier(n_estimators=100, random_state=42)
#rf.fit(X, y)
#rf_importance = pd.Series(rf.feature_importances_, index=X.columns)

# 2. Combine and Compare Strengths
importance_df = pd.DataFrame({
    'Correlation': correlations,
    'Mutual_Info': mi_series,
    'Univariate_F_Score': f_series
}).sort_values(by='Univariate_F_Score', ascending=False)

print("Comprehensive Predictor Strength Assessment:")
print(importance_df)

We're choosin no_of_special_requests, lead_time, arrival_year, no_of_week_nights, no_of_adults, avg_price_per_room, market_segment_type, repeated_guest the reasons being their values rank highgest in our strength predictor assesment.

In [ ]:
# mean target encoding function
def mean_target_encoding(dt, predictor, target, alpha = 0.01):
    total_cnt = len(dt)
    total_dr = np.mean(dt[target])
    dt_grp = dt.groupby(predictor).agg(
        categ_dr = (target, 'mean'),
        categ_cnt = (target, len)
    )

    dt_grp['categ_freq'] = dt_grp['categ_cnt'] / total_cnt
    dt_grp['categ_encoding'] = (dt_grp['categ_freq'] * dt_grp['categ_dr'] + alpha * total_dr) / (dt_grp['categ_freq'] + alpha)

    return dt_grp[['categ_encoding']].to_dict()['categ_encoding']

selected_features = [
    'no_of_special_requests', 'lead_time', 'arrival_year',
    'no_of_week_nights', 'no_of_adults', 'avg_price_per_room',
    'market_segment_type', 'repeated_guest'
]
target_col = 'booking_status'

# Drop rows where the target variable is missing
df_clean = df.dropna(subset=[target_col]).copy()

# 3. Imputation (Handling Missing Values)
# We use 'median' for numbers to avoid outlier bias and 'most_frequent' for text
num_cols = ['no_of_special_requests', 'lead_time', 'arrival_year',
            'no_of_week_nights', 'no_of_adults', 'avg_price_per_room', 'repeated_guest']
cat_cols = ['market_segment_type']

num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

df_clean[num_cols] = num_imputer.fit_transform(df_clean[num_cols])
df_clean[cat_cols] = cat_imputer.fit_transform(df_clean[cat_cols])

# 4. Variable Encoding
# Apply Target Encoding to 'market_segment_type'
encoding_map = mean_target_encoding(df_clean, 'market_segment_type', target_col, alpha=0.01)
df_clean['market_segment_encoded'] = df_clean['market_segment_type'].map(encoding_map)

# 5. Numerical Transformations
# Apply Log Transform to 'lead_time' and 'avg_price_per_room' (Skewed variables)
# We use log1p to safely handle zero values: log(1 + x)
df_clean['lead_time_log'] = np.log1p(df_clean['lead_time'])
df_clean['avg_price_per_room_log'] = np.log1p(df_clean['avg_price_per_room'])

# 6. Final Feature Selection
# Retain only the transformed and relevant columns
X = df_clean[[
    'no_of_special_requests', 'lead_time_log', 'arrival_year',
    'no_of_week_nights', 'no_of_adults', 'avg_price_per_room_log',
    'market_segment_encoded', 'repeated_guest'
]]
y = df_clean[target_col]

# 7. Standardization (Scaling)
# This centers all data at 0 with a standard deviation of 1
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# 8. Combine into a final cleaned training set
df_final = pd.concat([X_scaled, y.reset_index(drop=True)], axis=1)

# Display the result
print("Final Preprocessed Features:")
print(df_final.head())

Encode categorial variables using mean target encoding function(cviko)

In [ ]:
params_grid = [
    {
        'max_depth': [10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'criterion': ['gini', 'entropy']
    }
]

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=params_grid,
    cv=3,
    scoring='f1'
)
grid_search.fit(X, y)
print(f"Best Grid Search Params: {grid_search.best_params_}")

In [ ]:
params_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_leaf': randint(1, 5)
}

halving_search = HalvingRandomSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=params_dist,
    n_candidates=20,    # <--- Add this! (e.g., try 20 different model settings)
    factor=2,
    resource='n_samples',
    min_resources='exhaust', # Now 'exhaust' will work because it knows n_candidates
    cv=3,
    scoring='f1',
    random_state=42
)
halving_search.fit(X, y)
print(halving_search.best_params_)
#dasa